In [17]:
import pandas as pd
import numpy as np

import requests
import os
from dotenv import load_dotenv

import concurrent.futures
import threading


In [18]:
# load spotify data
df = pd.read_csv(r'excel_data\Alltime_Spotify_Streaming_Data.csv', low_memory=False)

print(f"Loaded Database: {df.shape[0]} rows, {df.shape[1]} columns")


Loaded Database: 517209 rows, 22 columns


In [19]:
df

,time_stamp,streaming_platform,ms_played,country_streamed,track_title,track_album_name,track_album_name2,track_spotify_url,episode_name,episode_show_name,...,audiobook_url,audiobook_chapter_url,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2016-01-24T19:10:33Z,"Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G...",6154,US,Tear in My Heart,Twenty One Pilots,Blurryface,spotify:track:3bnVBN67NBEzedqQuWrpP4,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,True,False,NaN,False
1,2016-01-24T22:08:36Z,"Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G...",240754,US,Fun (feat. Tove Lo),Coldplay,A Head Full of Dreams,spotify:track:7fJFDK6XjYsXcMKNHESbot,NaN,NaN,...,NaN,NaN,NaN,appload,trackdone,False,False,False,NaN,False
2,2016-01-24T22:09:56Z,"Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G...",73585,US,Someone Like You,Adele,21,spotify:track:4kflIGfjdZJW4ot2ioixTB,NaN,NaN,...,NaN,NaN,NaN,trackdone,endplay,False,True,False,NaN,False
3,2016-01-24T22:13:06Z,"Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G...",189386,US,Immortals,Fall Out Boy,American Beauty/American Psycho,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,NaN,NaN,...,NaN,NaN,NaN,clickrow,trackdone,False,False,False,NaN,False
4,2016-01-24T22:17:10Z,"Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G...",243353,US,Scary Monsters and Nice Sprites,Skrillex,Scary Monsters and Nice Sprites EP,spotify:track:4rwpZEcnalkuhPyGkEdhu0,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,False,False,False,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517204,2025-11-28T23:48:48Z,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517205,2025-11-28T23:51:13Z,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517206,2025-11-28T23:53:39Z,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517207,2025-11-28T23:56:04Z,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False


In [20]:
############################
# STEP 1: HEADERS CLEANING #
############################

df.columns = df.columns.str.replace('_', ' ')

df = df.rename(columns={
    'track album name': 'track artist',
    'track album name2': 'track album name'
})

print(df.columns)


Index(['time stamp', 'streaming platform', 'ms played', 'country streamed',
       'track title', 'track artist', 'track album name', 'track spotify url',
       'episode name', 'episode show name', 'episode spotify url',
       'audiobook title', 'audiobook url', 'audiobook chapter url',
       'audiobook chapter title', 'reason start', 'reason end', 'shuffle',
       'skipped', 'offline', 'offline timestamp', 'incognito mode'],
      dtype='object')


In [21]:
#############################
# STEP 2: CATEGORICAL TYPOS #
#############################

cleanup_map = {
    'Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'iOS 9.2.1 (iPad2,2)': 'iPad2',
    'Android OS 6.0.1 API 23 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android OS 7.0 API 24 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android-tablet OS 7.0 API 24 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android OS 6.0.1 API 23 (HUAWEI, KIW-L24)': 'KIW-L24',
    'Android OS 8.0.0 API 26 (motorola, moto g(6) play)': 'moto g(6) play',
    'Android OS 9 API 28 (motorola, moto g(6) play)': 'moto g(6) play',
    'Android OS 8.0.0 API 26 (samsung, SM-G950U)': 'SM-G950U',
    'Android OS 9 API 28 (samsung, SM-G950U)': 'SM-G950U',
    'Android OS 11 API 30 (samsung, SM-G781U)': 'SM-G781U',
    'Android OS 12 API 31 (samsung, SM-G781U)': 'SM-G781U',
    'not_applicable': np.nan # handle ghost values     
}

df['streaming platform'] = df['streaming platform'].replace(cleanup_map)

print(df['streaming platform'].unique())

['SAMSUNG-SM-G920A' 'iPad2' 'Windows 10 (10.0.14393; x64)'
 'Windows 10 (10.0.15063; x64)' 'Windows 10 (10.0.16299; x64)' 'KIW-L24'
 'moto g(6) play' 'Windows 10 (10.0.17134; x64; AppX)'
 'Partner google cast_tv;BRAVIA_4K_GB;;2.0.0-671-be56159--1.36.151708'
 'Windows 10 (10.0.17763; x64; AppX)'
 'Partner google cast_tv;BRAVIA_4K_GB;;2.0.0-671-be56159--1.36.154871'
 'Partner google cast_tv;BRAVIA_4K_GB;;2.2.1-7d79c17--1.36.154871'
 'SM-G950U' 'Partner google cast_tv;BRAVIA_4K_GB;;3.1.1--1.42.179832'
 'Partner android_tv Sony;BRAVIA4KGB;756a522d9f1648b89e76e80be654456a;;tpapi'
 'Partner google cast_tv;BRAVIA_4K_GB;;3.2.4--1.47.207274'
 'Partner google cast_tv;BRAVIA_4K_GB;;3.3.0--1.47.211678'
 'web_player windows 10;microsoft edge 85.0.564.68;desktop'
 'web_player windows 10;chrome 91.0.4472.77;desktop'
 'Partner roku_tv roku;4200x;4916bf2fd1c54ff2bace038314d21f39;;tpapi'
 'SM-G781U' 'web_player osx 10.15.7;safari 15.2;desktop'
 'Windows 10 (10.0.19044; x64; AppX)' 'android' 'windows' 'o

In [22]:
###############################
# STEP 3: DATE TIME DATA TYPE #
###############################

print(df['time stamp'].dtype)

df['time stamp'] = pd.to_datetime(df['time stamp'],errors = 'coerce')

df['time stamp']

object


0        2016-01-24 19:10:33+00:00
1        2016-01-24 22:08:36+00:00
2        2016-01-24 22:09:56+00:00
3        2016-01-24 22:13:06+00:00
4        2016-01-24 22:17:10+00:00
                    ...           
517204   2025-11-28 23:48:48+00:00
517205   2025-11-28 23:51:13+00:00
517206   2025-11-28 23:53:39+00:00
517207   2025-11-28 23:56:04+00:00
517208   2025-11-28 23:58:29+00:00
Name: time stamp, Length: 517209, dtype: datetime64[ns, UTC]

In [23]:
###################################
# STEP 4: LOGICAL INTEGRITY CHECK #
#      A: (No Start or End)       #
###################################

impossible_track1 = df['reason start'].isna()
impossible_track2 = df['reason end'].isna()

print(df.loc[impossible_track1])
print(df.loc[impossible_track2])

Empty DataFrame
Columns: [time stamp, streaming platform, ms played, country streamed, track title, track artist, track album name, track spotify url, episode name, episode show name, episode spotify url, audiobook title, audiobook url, audiobook chapter url, audiobook chapter title, reason start, reason end, shuffle, skipped, offline, offline timestamp, incognito mode]
Index: []

[0 rows x 22 columns]
Empty DataFrame
Columns: [time stamp, streaming platform, ms played, country streamed, track title, track artist, track album name, track spotify url, episode name, episode show name, episode spotify url, audiobook title, audiobook url, audiobook chapter url, audiobook chapter title, reason start, reason end, shuffle, skipped, offline, offline timestamp, incognito mode]
Index: []

[0 rows x 22 columns]


In [24]:
###################################
# STEP 4: LOGICAL INTEGRITY CHECK #
#      b: (less Than 0 ms Played) #
###################################
# step 4b: check logical integrity (less than 0 ms played)
impossible_track3 = df['ms played'] < 0

print(df.loc[impossible_track3])

Empty DataFrame
Columns: [time stamp, streaming platform, ms played, country streamed, track title, track artist, track album name, track spotify url, episode name, episode show name, episode spotify url, audiobook title, audiobook url, audiobook chapter url, audiobook chapter title, reason start, reason end, shuffle, skipped, offline, offline timestamp, incognito mode]
Index: []

[0 rows x 22 columns]


In [26]:
#######################################
# STEP 5: REMOVE UNNECESSARY ROWS     #
#      A: (Tracks that are not songs) #
#######################################

# check if there are podcast or audiobook tracks
not_songs = df['episode name'].notna() | df['audiobook title'].notna()
# print(df.loc[not_songs])

# remove these tracks from dataframe
df = df[~not_songs]

df

,time stamp,streaming platform,ms played,country streamed,track title,track artist,track album name,track spotify url,episode name,episode show name,...,audiobook url,audiobook chapter url,audiobook chapter title,reason start,reason end,shuffle,skipped,offline,offline timestamp,incognito mode
0,2016-01-24 19:10:33+00:00,SAMSUNG-SM-G920A,6154,US,Tear in My Heart,Twenty One Pilots,Blurryface,spotify:track:3bnVBN67NBEzedqQuWrpP4,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,True,False,NaN,False
1,2016-01-24 22:08:36+00:00,SAMSUNG-SM-G920A,240754,US,Fun (feat. Tove Lo),Coldplay,A Head Full of Dreams,spotify:track:7fJFDK6XjYsXcMKNHESbot,NaN,NaN,...,NaN,NaN,NaN,appload,trackdone,False,False,False,NaN,False
2,2016-01-24 22:09:56+00:00,SAMSUNG-SM-G920A,73585,US,Someone Like You,Adele,21,spotify:track:4kflIGfjdZJW4ot2ioixTB,NaN,NaN,...,NaN,NaN,NaN,trackdone,endplay,False,True,False,NaN,False
3,2016-01-24 22:13:06+00:00,SAMSUNG-SM-G920A,189386,US,Immortals,Fall Out Boy,American Beauty/American Psycho,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,NaN,NaN,...,NaN,NaN,NaN,clickrow,trackdone,False,False,False,NaN,False
4,2016-01-24 22:17:10+00:00,SAMSUNG-SM-G920A,243353,US,Scary Monsters and Nice Sprites,Skrillex,Scary Monsters and Nice Sprites EP,spotify:track:4rwpZEcnalkuhPyGkEdhu0,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,False,False,False,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517204,2025-11-28 23:48:48+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517205,2025-11-28 23:51:13+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517206,2025-11-28 23:53:39+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517207,2025-11-28 23:56:04+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False


In [28]:
#######################################
# STEP 5: REMOVE UNNECESSARY ROWS     #
#      B: (Tracks played for 0 ms)    #
#######################################

# check if there are tracks played for 0 seconds
zero_ms = df['ms played'] == 0
# print(df.loc[zero_ms])

# remove these tracks from dataframe
df = df[~zero_ms]

df

,time stamp,streaming platform,ms played,country streamed,track title,track artist,track album name,track spotify url,episode name,episode show name,...,audiobook url,audiobook chapter url,audiobook chapter title,reason start,reason end,shuffle,skipped,offline,offline timestamp,incognito mode
0,2016-01-24 19:10:33+00:00,SAMSUNG-SM-G920A,6154,US,Tear in My Heart,Twenty One Pilots,Blurryface,spotify:track:3bnVBN67NBEzedqQuWrpP4,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,True,False,NaN,False
1,2016-01-24 22:08:36+00:00,SAMSUNG-SM-G920A,240754,US,Fun (feat. Tove Lo),Coldplay,A Head Full of Dreams,spotify:track:7fJFDK6XjYsXcMKNHESbot,NaN,NaN,...,NaN,NaN,NaN,appload,trackdone,False,False,False,NaN,False
2,2016-01-24 22:09:56+00:00,SAMSUNG-SM-G920A,73585,US,Someone Like You,Adele,21,spotify:track:4kflIGfjdZJW4ot2ioixTB,NaN,NaN,...,NaN,NaN,NaN,trackdone,endplay,False,True,False,NaN,False
3,2016-01-24 22:13:06+00:00,SAMSUNG-SM-G920A,189386,US,Immortals,Fall Out Boy,American Beauty/American Psycho,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,NaN,NaN,...,NaN,NaN,NaN,clickrow,trackdone,False,False,False,NaN,False
4,2016-01-24 22:17:10+00:00,SAMSUNG-SM-G920A,243353,US,Scary Monsters and Nice Sprites,Skrillex,Scary Monsters and Nice Sprites EP,spotify:track:4rwpZEcnalkuhPyGkEdhu0,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,False,False,False,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517204,2025-11-28 23:48:48+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517205,2025-11-28 23:51:13+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517206,2025-11-28 23:53:39+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False
517207,2025-11-28 23:56:04+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1.764374e+09,False


In [29]:
######################################
# STEP 6: REMOVE UNNECESSARY COLUMNS #
######################################

df = df.drop(['episode name', 'episode show name', 'episode spotify url', 
              'audiobook title', 'audiobook url', 'audiobook chapter url',
              'audiobook chapter title', 'offline timestamp'],axis=1)

df

,time stamp,streaming platform,ms played,country streamed,track title,track artist,track album name,track spotify url,reason start,reason end,shuffle,skipped,offline,incognito mode
0,2016-01-24 19:10:33+00:00,SAMSUNG-SM-G920A,6154,US,Tear in My Heart,Twenty One Pilots,Blurryface,spotify:track:3bnVBN67NBEzedqQuWrpP4,clickrow,endplay,False,True,False,False
1,2016-01-24 22:08:36+00:00,SAMSUNG-SM-G920A,240754,US,Fun (feat. Tove Lo),Coldplay,A Head Full of Dreams,spotify:track:7fJFDK6XjYsXcMKNHESbot,appload,trackdone,False,False,False,False
2,2016-01-24 22:09:56+00:00,SAMSUNG-SM-G920A,73585,US,Someone Like You,Adele,21,spotify:track:4kflIGfjdZJW4ot2ioixTB,trackdone,endplay,False,True,False,False
3,2016-01-24 22:13:06+00:00,SAMSUNG-SM-G920A,189386,US,Immortals,Fall Out Boy,American Beauty/American Psycho,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,clickrow,trackdone,False,False,False,False
4,2016-01-24 22:17:10+00:00,SAMSUNG-SM-G920A,243353,US,Scary Monsters and Nice Sprites,Skrillex,Scary Monsters and Nice Sprites EP,spotify:track:4rwpZEcnalkuhPyGkEdhu0,trackdone,trackdone,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517204,2025-11-28 23:48:48+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,trackdone,trackdone,True,False,False,False
517205,2025-11-28 23:51:13+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,trackdone,trackdone,True,False,False,False
517206,2025-11-28 23:53:39+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,trackdone,trackdone,True,False,False,False
517207,2025-11-28 23:56:04+00:00,windows,144522,US,Fever,Buckshot,Fever,spotify:track:09xhawlPUifhftf8zuie7w,trackdone,trackdone,True,False,False,False


In [30]:
######################################
# STEP 7: INDENTIFY UNIQUE TRACKS    #
######################################

df_unique = df.drop_duplicates(subset=['track spotify url'])
df_unique = df_unique[['track title', 'track artist', 'track spotify url']]

# determine number of unique songs
num_unique_songs = df_unique.shape[0]
user_message = f"You have listened to {num_unique_songs} different tracks in the last 10 years!"
print(user_message)

# save list of unique songs as csv file
df_unique.to_csv('unique_songs.csv', index=False)

You have listened to 27434 different tracks in the last 10 years!


In [31]:
#########################################
# STEP 8a: CLEAN CSV TO UPLOAD NEW DATA #
#########################################


df2 = pd.read_csv('unique_songs.csv')
df2['spotify track id'] = df2['track spotify url'].str.split(':', n=2).str[-1]

# create new columns for api call variables
added_columns = [
    "id", "key", "mode", "camelot", "tempo", "duration",
    "popularity", "energy", "danceability", "happiness",
    "acousticness", "instrumentalness", "liveness",
    "speechiness", "loudness"
]

for col in added_columns:
    df2[col] = None

df2


,track title,track artist,track spotify url,spotify track id,id,key,mode,camelot,tempo,duration,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness
0,Tear in My Heart,Twenty One Pilots,spotify:track:3bnVBN67NBEzedqQuWrpP4,3bnVBN67NBEzedqQuWrpP4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,Fun (feat. Tove Lo),Coldplay,spotify:track:7fJFDK6XjYsXcMKNHESbot,7fJFDK6XjYsXcMKNHESbot,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,Someone Like You,Adele,spotify:track:4kflIGfjdZJW4ot2ioixTB,4kflIGfjdZJW4ot2ioixTB,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,Immortals,Fall Out Boy,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,3Te8uLyit6X3ncNW8Fp3K2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,Scary Monsters and Nice Sprites,Skrillex,spotify:track:4rwpZEcnalkuhPyGkEdhu0,4rwpZEcnalkuhPyGkEdhu0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27429,in the sea,kensuke ushio,spotify:track:6hShfxTdYcC7YQ3AhfRedv,6hShfxTdYcC7YQ3AhfRedv,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27430,ジェーンは教会で眠った,レゼ（上田麗奈）,spotify:track:1YS1h55ne33IOw3u1CG2UD,1YS1h55ne33IOw3u1CG2UD,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27431,JANE DOE,Kenshi Yonezu,spotify:track:4oE7MyJhqSD3BaHRpNs8Nl,4oE7MyJhqSD3BaHRpNs8Nl,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27432,Garden Waltz,Alstad,spotify:track:4wuQUHu2WQu3jSkK9wCRFY,4wuQUHu2WQu3jSkK9wCRFY,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [32]:
# check how long api call takes = 457 ms ± 110 ms per loop
%timeit session.get(f"https://track-analysis.p.rapidapi.com/pktx/spotify/{df2.at[0,'spotify track id']}")

407 ms ± 38.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
#########################################
# STEP 8b: ITERATE API CALLS            #
#########################################
load_dotenv()
api_key = os.getenv("RAPIDAPI_KEY")

# initialize lock
lock = threading.Lock()

# set up api call session
headers = {
        'x-rapidapi-key': api_key,
        'x-rapidapi-host': 'track-analysis.p.rapidapi.com'
    }
session = requests.Session()
session.headers.update(headers)

# function get_api_data
def get_api_data(i):
    # set spotifyId based on index i
    spotifyId = df2.at[i,'spotify track id']
    
    # configure api request
    url = f"https://track-analysis.p.rapidapi.com/pktx/spotify/{spotifyId}"
    
    # make api call
    result = session.get(url)
    
    # error check
    if result.status_code == 200:
        output = result.json()
    else:
        output = {}
        print(f"Error {result.status_code}: \n{result.text}")
        
    # thread safety to avoid corrupt data
    with lock:
        for col in added_columns:
            df2.at[i,col] = output.get(col)
            
    # debugging setup
    return i


# run get_api_data for all tracks, 5 at a time
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(get_api_data, i) for i in range(0,10)]
    
    for count, task in enumerate(concurrent.futures.as_completed(futures)):
        task.result()
        if count % 10 == 0:
            print(f"Processed {count} tracks.")

# signal end of api calls
print("Finished processing all tracks!")

df2



Processed 0 tracks.
Error 429: 
{"message":"You have exceeded the rate limit per second for your plan, PRO, by the API provider"}
Error 429: 
{"message":"You have exceeded the rate limit per second for your plan, PRO, by the API provider"}
Error 429: 
{"message":"You have exceeded the rate limit per second for your plan, PRO, by the API provider"}
Error 429: 
{"message":"You have exceeded the rate limit per second for your plan, PRO, by the API provider"}
Error 429: 
{"message":"You have exceeded the rate limit per second for your plan, PRO, by the API provider"}
Finished processing all tracks!


,track title,track artist,track spotify url,spotify track id,id,key,mode,camelot,tempo,duration,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness
0,Tear in My Heart,Twenty One Pilots,spotify:track:3bnVBN67NBEzedqQuWrpP4,3bnVBN67NBEzedqQuWrpP4,dGVhci1pbi1teS1oZWFydC10d2VudHktb25lLXBpbG90cw==,D,major,10B,120,3:08,78,63,66,45,2,0,7,5,-5 dB
1,Fun (feat. Tove Lo),Coldplay,spotify:track:7fJFDK6XjYsXcMKNHESbot,7fJFDK6XjYsXcMKNHESbot,ZnVuLWZlYXQtdG92ZS1sby1jb2xkcGxheQ==,C,major,8B,134,4:28,58,74,49,22,27,0,9,4,-5 dB
2,Someone Like You,Adele,spotify:track:4kflIGfjdZJW4ot2ioixTB,4kflIGfjdZJW4ot2ioixTB,c29tZW9uZS1saWtlLXlvdS1hZGVsZQ==,A,major,11B,135,4:45,80,32,56,29,89,0,10,3,-8 dB
3,Immortals,Fall Out Boy,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,3Te8uLyit6X3ncNW8Fp3K2,aW1tb3J0YWxzLWZhbGwtb3V0LWJveQ==,D,major,10B,108,3:09,79,87,62,48,0,0,67,7,-4 dB
4,Scary Monsters and Nice Sprites,Skrillex,spotify:track:4rwpZEcnalkuhPyGkEdhu0,4rwpZEcnalkuhPyGkEdhu0,c2NhcnktbW9uc3RlcnMtYW5kLW5pY2Utc3ByaXRlcy1za3...,G,minor,6A,140,4:03,65,94,52,32,0,56,12,8,-4 dB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27429,in the sea,kensuke ushio,spotify:track:6hShfxTdYcC7YQ3AhfRedv,6hShfxTdYcC7YQ3AhfRedv,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27430,ジェーンは教会で眠った,レゼ（上田麗奈）,spotify:track:1YS1h55ne33IOw3u1CG2UD,1YS1h55ne33IOw3u1CG2UD,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27431,JANE DOE,Kenshi Yonezu,spotify:track:4oE7MyJhqSD3BaHRpNs8Nl,4oE7MyJhqSD3BaHRpNs8Nl,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27432,Garden Waltz,Alstad,spotify:track:4wuQUHu2WQu3jSkK9wCRFY,4wuQUHu2WQu3jSkK9wCRFY,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [ ]:
#########################################
# STEP 9: LOAD API DATA INTO CSV FILE   #
#########################################

df2.to_csv('rapidapi_spotify_track_data.csv', index=False)